In [43]:
# Cell 1: Import necessary libraries

import os
import cv2
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tf2onnx
import onnx


In [ ]:
# Cell 2: Set paths and directories

IMAGE_FOLDER_PROVIDEDIMAGES = r"C:\Users\beren\Documents\paparazzi\playground\provided_images"
IMAGE_FOLDER_VIDEOCAPSIMULATIONROUND1 = r"C:\Users\beren\Documents\paparazzi\playground\videocap_simulation_round1"
IMAGE_FOLDER_TESTROUND2 = r"C:\Users\beren\Documents\paparazzi\playground\videocap_testround2"

LABELS_JSON_PROVIDEDIMAGES = r"C:\Users\beren\Documents\paparazzi\playground\labelled_provided_images\results.json"
LABELS_JSON_VIDEOCAPSIMULATIONROUND1 = r"C:\Users\beren\Documents\paparazzi\playground\labelled_videocap_simulation_round1\results.json"
LABELS_JSON_TESTROUND2 = r"C:\Users\beren\Documents\paparazzi\playground\labelled_videocap_testround2\results.json"

OUTPUT_FOLDER = "CNN_Output"
IMAGE_FOLDER = r"C:\Users\beren\Documents\paparazzi\playground\labelled_videocap_simulation_round1\results.json"
VIDEO_OUTPUT =  "CNN_Video"
TEXT_OUTPUT =  "CNN_Text"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)


SyntaxError: invalid syntax (4239386741.py, line 12)

In [ ]:
# Cell 3: Load labels from JSON file
labels_data = {}

with open(LABELS_JSON_PROVIDEDIMAGES, "r") as f:
    data = json.load(f)
    labels_data["provided_images"] = data
with open(LABELS_JSON_VIDEOCAPSIMULATIONROUND1, "r") as f:
    data = json.load(f)
    labels_data["videocap_simulation_round1"] = data
with open(LABELS_JSON_TESTROUND2, "r") as f:
    data = json.load(f)
    labels_data["test_round2"] = data

In [ ]:

# Function to load data and save first image as a text file
def load_data(image_folder, labels, dataset_name, img_size=(520, 240)):
    images, label_grids = [], []
    
    for i, (filename, data) in enumerate(labels.items()):
        img_path = os.path.join(image_folder, f"{filename}.jpg")
        if not os.path.exists(img_path):
            continue

        # Read the image and convert from BGR to YUV
        img = cv2.imread(img_path)
        img_yuv = cv2.cvtColor(img, cv2.COLOR_BGR2YUV)
        images.append(img_yuv)

        # Save the first image's YUV values to a separate text file for each dataset
        if i == 0:
            yuv_output_path = os.path.join(TEXT_OUTPUT, f"{dataset_name}_first_image_yuv.txt")
            with open(yuv_output_path, "w") as f:
                f.write(f"{img_yuv.tolist()}\n")
            print(f"First YUV image grid saved to {yuv_output_path}")

        # Get the label grid and append it
        label_grid = np.array(data["scores"])
        label_grids.append(label_grid)

    return np.array(images), np.array(label_grids)

# Load datasets and save first images
X_VIDEOCAPSIMULATIONROUND1, y_VIDEOCAPSIMULATIONROUND1 = load_data(
    IMAGE_FOLDER_VIDEOCAPSIMULATIONROUND1, labels_data["videocap_simulation_round1"], "videocap_simulation_round1"
)
y_VIDEOCAPSIMULATIONROUND1 = np.expand_dims(y_VIDEOCAPSIMULATIONROUND1, axis=-1)

X_TESTROUND2, y_TESTROUND2 = load_data(
    IMAGE_FOLDER_TESTROUND2, labels_data["test_round2"], "test_round2"
)
y_TESTROUND2 = np.expand_dims(y_TESTROUND2, axis=-1)

X_PROVIDEDIMAGES, y_PROVIDEDIMAGES = load_data(
    IMAGE_FOLDER_PROVIDEDIMAGES, labels_data["provided_images"], "provided_images"
)
y_PROVIDEDIMAGES = np.expand_dims(y_PROVIDEDIMAGES, axis=-1)

print(X_VIDEOCAPSIMULATIONROUND1.shape, X_TESTROUND2.shape)

# Concatenate all datasets into training data
X_TRAIN = np.concatenate([X_VIDEOCAPSIMULATIONROUND1, X_TESTROUND2], axis=0)
y_TRAIN = np.concatenate([y_VIDEOCAPSIMULATIONROUND1, y_TESTROUND2], axis=0)

X_TEST = X_PROVIDEDIMAGES
y_TEST = y_PROVIDEDIMAGES


First YUV image grid saved to CNN_Text\videocap_simulation_round1_first_image_yuv.txt
First YUV image grid saved to CNN_Text\test_round2_first_image_yuv.txt
First YUV image grid saved to CNN_Text\provided_images_first_image_yuv.txt
(1588, 240, 520, 3) (1998, 240, 520, 3)


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import models, layers
from tensorflow.keras.backend import int_shape


def build_model(optimizer='adam',
                conv1_filters=16, conv1_stride_y=32, conv1_stride_x=32,
                conv2_filters=32, conv3_filters=64):
    model = models.Sequential([
        layers.Conv2D(conv1_filters, (1, 1),
                      strides=(conv1_stride_y, conv1_stride_x),
                      activation='relu',
                      input_shape=(240, 520, 3)),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(conv2_filters, (1, 1), strides=(1, 1), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(conv3_filters, (1, 1), strides=(1, 1), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),

        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(y_TRAIN.shape[1] * y_TRAIN.shape[2],
                     activation='sigmoid', name='output')
    ])

    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    return model

# Build the model
model = build_model(optimizer='adam')

# Run a dummy forward pass to build the model and set layer shapes
dummy_input = np.random.randn(1, 240, 520, 3)
_ = model(dummy_input)

def calculate_mac(model):
    total_mac = 0
    for layer in model.layers:
        if isinstance(layer, layers.Conv2D):
            # Use int_shape to get input and output shapes after model is built
            input_shape = int_shape(layer.input)
            output_shape = int_shape(layer.output)
            kernel_size = layer.kernel_size
            input_channels = input_shape[-1]
            output_filters = layer.filters
            output_h, output_w = output_shape[1], output_shape[2]
            mac = output_h * output_w * output_filters * kernel_size[0] * kernel_size[1] * input_channels
            total_mac += mac
        elif isinstance(layer, layers.Dense):
            input_units = int_shape(layer.input)[-1]
            output_units = layer.units
            mac = output_units * input_units
            total_mac += mac
    return total_mac

mac_count = calculate_mac(model)
print(f"Total MAC operations: {mac_count:,}")
model.summary()


Total MAC operations: 70,016


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 8, 17, 16)      │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 4, 8, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 4, 8, 32)       │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 2, 4, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 2, 4, 64)       │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 1, 2, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 96)             │         6,240 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,728 (131.75 KB)

 Trainable params: 33,728 (131.75 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Preprocessing

In [ ]:
## PLACEHOLDER: Data augmentation function
# Should run every epoch to randomize application of augmentation every epoch

# Operations:
# - Shuffle image order
# - rotation (which you did)
# - horizontal flipping
# - scaling (i.e. zooming)
# - photometrics (e.g. brightess, contrast, gamma)
# - blurring
# - random noise

In [ ]:
# Cell 7: Train the model with EarlyStopping callback

early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)

model.fit(X_TRAIN, y_TRAIN.reshape(y_TRAIN.shape[0], -1), epochs=20, batch_size=16, 
          validation_data=(X_TEST, y_TEST.reshape(y_TEST.shape[0], -1)),
          callbacks=[early_stopping])  


Epoch 1/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.1824 - mae: 0.3282 - val_loss: 0.0966 - val_mae: 0.2615
Epoch 2/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1040 - mae: 0.2600 - val_loss: 0.0966 - val_mae: 0.2616
Epoch 3/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0932 - mae: 0.2413 - val_loss: 0.0895 - val_mae: 0.2417
Epoch 4/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0893 - mae: 0.2322 - val_loss: 0.0906 - val_mae: 0.2404
Epoch 5/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0829 - mae: 0.2209 - val_loss: 0.0922 - val_mae: 0.2485
Epoch 6/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0806 - mae: 0.2168 - val_loss: 0.0925 - val_mae: 0.2458
Epoch 7/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0789 - mae: 0.2135 - val_loss: 0.0923 - val_mae: 0.2386
Epoch 8/20
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0730 - mae: 0.2022 - val_loss: 0.0916 - val_mae: 0.2371
Epoch 8: early stopping
Restoring model weights from the

In [ ]:
# Cell 8: Make predictions

y_pred = model.predict(X_TEST)
y_pred = y_pred.reshape(y_TEST.shape)  # Reshape to grid format
print(y_pred.shape)


15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
(458, 6, 16, 1)


In [ ]:
# Cell 9: Overlay safety grid onto images

def overlay_safety_grid(base_img, safety_grid, alpha=0.4):
    overlay = base_img.copy()
    h, w, _ = overlay.shape
    grid_height, grid_width, extra = safety_grid.shape
    cell_h = h // grid_height
    cell_w = w // grid_width

    for gy in range(grid_height):
        for gx in range(grid_width):
            val = float(safety_grid[gy, gx])  
            r = int((1.0 - val) * 255)
            g = int(val * 255)
            b = 0
            color = (b, g, r)
            
            y_start = gy * cell_h
            y_end   = (gy+1) * cell_h if gy < grid_height - 1 else h
            x_start = gx * cell_w
            x_end   = (gx+1) * cell_w if gx < grid_width - 1 else w

            cv2.rectangle(overlay, (x_start, y_start), (x_end, y_end), color, -1)

    blended = cv2.addWeighted(overlay, alpha, base_img, 1 - alpha, 0)
    return blended


In [ ]:
# Cell 10: Save predicted images

def save_predicted_images(X_TEST, y_pred, y_TEST, output_folder):
    for i in range(len(X_TEST)):
        original = (X_TEST[i] * 255).astype(np.uint8)
        pred_overlay = overlay_safety_grid(original, y_pred[i])
        gt_overlay = overlay_safety_grid(original, y_TEST[i])
        
        combined = np.hstack((gt_overlay, pred_overlay))  # Side-by-side comparison
        cv2.imwrite(os.path.join(output_folder, f"pred_{i}.jpg"), combined)

save_predicted_images(X_TEST, y_pred, y_TEST, OUTPUT_FOLDER)


C:\Users\beren\AppData\Local\Temp\ipykernel_29172\937759760.py:12: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  val = float(safety_grid[gy, gx])


In [ ]:
# Cell 11: Generate video from saved images

def create_video(image_folder, output_video, fps=1):
    images = [img for img in os.listdir(image_folder) if img.endswith(".jpg")]
    images.sort()
    
    if not images:
        print("No images found to create video.")
        return
    
    frame = cv2.imread(os.path.join(image_folder, images[0]))
    h, w, _ = frame.shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video = cv2.VideoWriter(output_video, fourcc, fps, (w, h))
    
    for image in images:
        frame = cv2.imread(os.path.join(image_folder, image))
        video.write(frame)
    
    video.release()
    print(f"Video saved to {output_video}")

create_video(OUTPUT_FOLDER, VIDEO_OUTPUT)


Video saved to CNN_Video


In [ ]:
# Cell 12: Save and convert the model to ONNX format

model.output_names=['output']

# Save the Keras model
model.save(r"C:\Users\beren\Documents\paparazzi\playground\cnn_model.keras")

# Convert the trained model to ONNX format
onnx_model_path = r"C:\Users\beren\Documents\paparazzi\playground\cnn_model.onnx"
onnx_model, _ = tf2onnx.convert.from_keras(
    model,
    opset=13,
    input_signature=[tf.TensorSpec(shape=[None, 240, 520, 3], dtype=tf.float32)],
)

with open(onnx_model_path, "wb") as f:
    f.write(onnx_model.SerializeToString())

print("Model successfully converted to ONNX format!")


Model successfully converted to ONNX format!


In [ ]:
# Cell 13: Verify the ONNX model

onnx_model = onnx.load(onnx_model_path)
onnx.checker.check_model(onnx_model)
print("ONNX model is valid.")


ONNX model is valid.


In [ ]:
# Cell 14: Test model on the first image in the folder

image_files = sorted([f for f in os.listdir(IMAGE_FOLDER) if f.endswith(".jpg")])
if not image_files:
    print("No images found in the folder.")
else:
    first_image_path = os.path.join(IMAGE_FOLDER, image_files[100])

    # Load and preprocess the image
    first_img = cv2.imread(first_image_path)
    first_img_yuv = cv2.cvtColor(first_img, cv2.COLOR_BGR2YUV)
    first_img_yuv = np.expand_dims(first_img_yuv, axis=0)

    # Predict
    first_pred = model.predict(first_img_yuv)
    first_pred = first_pred.reshape(y_train.shape[1:])

    # Save the safety grid of the first image
    safety_grid_output_path = os.path.join(TEXT_OUTPUT, "first_image_safety_grid.txt")
    with open(safety_grid_output_path, "w") as f:
        f.write(f"{first_pred.tolist()}\n")
    print(f"Safety grid of first image saved to {safety_grid_output_path}")

    # Overlay safety grid
    pred_overlay = overlay_safety_grid(first_img, first_pred)

    # Save or show the result
    output_first_pred_path = os.path.join(TEXT_OUTPUT, "first_image_prediction.jpg")
    cv2.imwrite(output_first_pred_path, pred_overlay)
    print(f"Prediction on first image saved to {output_first_pred_path}")
    
    # Display the image
    plt.imshow(cv2.cvtColor(pred_overlay, cv2.COLOR_BGR2RGB))
    plt.title("Predicted Safety Grid on First Image")
    plt.axis("off")
    plt.show()


NameError: name 'IMAGE_FOLDER' is not defined